# Cobblemon Atlas｜依遊戲 Poser 驗證烈空坐真正的浮空姿勢

直接讀取 `merged_pack/bedrock/pokemon/posers/0384_rayquaza/rayquaza.json`，列出全部姿勢、動畫引用與條件，挑出非戰鬥的飛行／浮空姿勢進行診斷渲染。所有輸出只放在 `Cobblemon_Atlas/reports/semantic-audit/game-poses/`，**不修改正式 WebP、checkpoint 或網站**。

注意：目前仍沒有完整支援 `q.look`、quirks 及條件式動畫；Notebook 會把未支援項目記錄下來，因此此測試仍不是完整遊戲等價渲染器。

In [ ]:
from pathlib import Path
import os,sys,shutil,subprocess,json,zipfile,time
DRIVE_ROOT=Path('/content/drive/MyDrive/Cobblemon_Atlas')
WORK_ROOT=Path('/content/cobblemon-bedrock')
SITE_ROOT=Path('/content/cobblemon-atlas')
BRANCH='feat/animation-manifest-scanner-phase1'
FORMS=['0384_rayquaza']
ACTIONS=['ground_walk']
FPS,SECONDS,SIZE,SS=12,2,224,2
print('Isolated Rayquaza test:',FORMS,ACTIONS)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert DRIVE_ROOT.is_dir(), f'找不到雲端資料夾 {DRIVE_ROOT}；請確認使用正確 Google 帳號'
for name in ['source_packs','manifests','renders','reports','checkpoints']:
    assert (DRIVE_ROOT/name).is_dir(), f'缺少 Drive 資料夾：{name}'
print('Google Drive 目錄確認成功:', DRIVE_ROOT)

In [ ]:
if not (SITE_ROOT/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch',BRANCH,'https://github.com/hwh0410/cobblemon-atlas.git',str(SITE_ROOT)],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'sparse-checkout','set','tools','docs','notebooks','mega-showdown/data'],check=True)
subprocess.run(['git','-C',str(SITE_ROOT),'pull','--ff-only','origin',BRANCH],check=True)
sha=subprocess.check_output(['git','-C',str(SITE_ROOT),'rev-parse','--short','HEAD'],text=True).strip()
renderer_source=(SITE_ROOT/'tools/render_action_jobs.py').read_text()
assert 'def choose_actions(' in renderer_source, '目前載入的是舊版選擇器，請重新開啟 GitHub 最新 Notebook'
print('已載入最新版候選來源選擇器，Git commit:',sha)

In [ ]:
# 使用本機暫存磁碟渲染，避免每一幀反覆從 Drive 讀取大量小檔案。
WORK_ROOT.mkdir(parents=True,exist_ok=True)
packs = DRIVE_ROOT/'source_packs'
required = [packs/'development_resource_packs.zip', packs/'merged_pack.zip', packs/'render_portraits.py', DRIVE_ROOT/'manifests/animation-actions.json']
missing = [str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('請先上傳以下來源檔：\n'+'\n'.join(missing))
for archive in required[:2]:
    expected = WORK_ROOT/archive.stem
    if expected.is_dir():
        print('Reuse extracted pack:', expected)
        continue
    print('Extracting:', archive.name, flush=True)
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            dest = (WORK_ROOT/item.filename).resolve()
            if not dest.is_relative_to(WORK_ROOT.resolve()): raise ValueError('Unsafe archive path')
        z.extractall(WORK_ROOT)
    assert expected.is_dir(), f'壓縮檔最外層不是 {archive.stem}，請重新封裝'
(WORK_ROOT/'tools').mkdir(exist_ok=True)
shutil.copy2(packs/'render_portraits.py', WORK_ROOT/'tools/render_portraits.py')
manifest = DRIVE_ROOT/'manifests/animation-actions.json'
data=json.loads(manifest.read_text())
print('Manifest:',data['summary'])
print('Resources prepared locally')

In [ ]:
# 實際讀取 Poser 定義與對應 Bedrock 動畫，並選擇遊戲定義的浮空姿勢。
tool=SITE_ROOT/'tools/render_rayquaza_game_poses.py'
assert tool.is_file(),'缺少最新診斷程式，請重新執行 Git pull 那格'
out=DRIVE_ROOT/'reports/semantic-audit/game-poses'
cmd=[sys.executable,str(tool),'--repo',str(WORK_ROOT),'--out',str(out)]
result=subprocess.run(cmd,check=False)
print('Exit code:',result.returncode)
if result.returncode:
    raise RuntimeError('Poser 姿勢實驗失敗，請將錯誤訊息貼回對話')
report=json.loads((out/'rayquaza-game-poses.json').read_text())
print('All in-game pose names:',list(report['all_pose_definitions']))
print('Chosen:',report['selected_poses'])
for name,item in report['renders'].items():
    print(name,item['clips'],'motion=',item['changed_pixels_max'],'bbox=',item['alpha_bbox'])
print('Reports and previews:',out)